# HCDE 530 — Week 5 In-Class Activity
### Five Questions — app_reviews_demo.csv

Answer each question using pandas. A plain-English comment above each code block explains what question it answers.

In [ ]:
import pandas as pd

df = pd.read_csv('app_reviews_demo.csv')

---
## Question 1 — What does your dataset look like?

In [ ]:
# What do the first rows look like? Are there any obvious issues?
df.head()

In [ ]:
df.tail()

In [ ]:
# What columns do we have, what type is each one, and how many non-null values are there?
df.info()

The dataset has 500 rows and 10 columns. Most columns are complete, but two have missing values: `device_type` (63 missing) and `app_version` (111 missing). The columns cover the app name, category, a 1–5 star rating, the review text, a date, helpful vote count, a verified purchase flag, the device type, and the app version. The mix of numeric, text, boolean, and categorical columns means different pandas operations will apply to different columns.

---
## Question 2 — What's the distribution of your most important column?

The most important column here is `rating` — it's the core signal in any app review dataset.

In [ ]:
# How are ratings distributed? Is the data balanced or skewed toward high/low scores?
counts = df['rating'].value_counts().sort_index()
pcts   = df['rating'].value_counts(normalize=True).sort_index().mul(100).round(1)
pd.concat([counts, pcts], axis=1, keys=['count', '%'])

Ratings are heavily skewed toward the positive end. 5-star reviews are the single largest group (207, 41.4%) and 4-star reviews are the second largest (160, 32%), meaning nearly three quarters of all reviews are positive. Negative reviews (1 and 2 stars) make up only about 14% combined. This kind of positivity skew is common in app review datasets and is worth keeping in mind — averages will be pulled upward, so a "low" average of 3.67 actually represents a meaningful drop from the norm.

---
## Question 3 — Filter to a meaningful subset. What's in it?

In [ ]:
# Which reviews are negative (rating below 3), and how much of the dataset are they?
negative = df[df['rating'] < 3]

print(f"{len(negative)} negative reviews ({len(negative)/len(df):.1%} of total)")
negative.head(10)

There are 72 negative reviews (ratings of 1 or 2), which is 14.4% of the full dataset. Fieldkit appears frequently in this subset — consistent with it having the lowest average rating overall (3.67). The negative reviews tend to mention specific friction points like slow search, missing features, and transcription accuracy issues. Even though this is a small slice of the data, it is often the most actionable for product teams because it surfaces concrete problems rather than general satisfaction.

---
## Question 4 — Group by a category and find the average of a numeric column.

In [ ]:
# Which app has the highest average rating, and how many reviews does each app have?
df.groupby('app')['rating'].agg(['mean', 'count']).round(2).sort_values('mean')

Dovetail has the highest average rating (4.12) and Fieldkit has the lowest (3.67), a gap of 0.45 stars. The five apps have similar review counts (89–121), so the differences in means are not simply due to one app having far fewer reviews. Miro has the most reviews (121) and sits near the middle (4.02). The spread between apps is relatively narrow — all means fall between 3.67 and 4.12 — but whether those differences are statistically meaningful requires a significance test.

---
## Question 5 — Where are the missing values? Are any columns incomplete?

In [ ]:
# How many values are missing per column, and what percentage of the column is that?
missing_count = df.isnull().sum()
missing_pct   = df.isnull().mean().mul(100).round(1)
summary = pd.concat([missing_count, missing_pct], axis=1, keys=['missing', '%'])
summary[summary['missing'] > 0]

Two columns have missing data: `device_type` is missing 63 values (12.6% of rows) and `app_version` is missing 111 values (22.2%). Every other column is complete. The `app_version` gap is significant — more than one in five rows — so any analysis that groups or filters by version should account for this. For `device_type`, 63 missing rows is manageable but not trivial; dropping them silently would remove 12.6% of the data, which could skew results if the missing rows are not random.